# Data gathering: rebuilding the open dataset from public sources

**Author:** Khaled Baha

This notebook rebuilds `data/open/MarketData_Normalized.csv` and `data/open/Ratings_Normalized.csv`, the 30-stock dataset used for the pre-registered out-of-sample test, from two public sources:

| Source | What | Licence |
|---|---|---|
| Kaggle dataset [`yijian2427/us-market-analyst-insights-with-full-stock-symbols`](https://www.kaggle.com/datasets/yijian2427/us-market-analyst-insights-with-full-stock-symbols) | broker-level analyst ratings and **target prices** with publication dates | MIT (as stated on the dataset page) |
| Yahoo Finance via `yfinance` | daily **split-adjusted closing prices** (`Close`, `auto_adjust=False`) | Yahoo's terms of use; personal / research use |

It runs in four steps:
1. download the Kaggle file and keep target prices for a fixed list of 30 tickers;
2. download matching closing prices;
3. save the raw tables to `data/raw/` (git-ignored: tickers and dollar levels are not redistributed);
4. anonymise and normalise them with `fmflow.data.normalize_universe` and **compare** the result with the shipped files.

The shipped files in `data/open/` are the frozen, pre-registered version and are never overwritten: the rebuilt files go to `data/rebuilt/`. Kaggle datasets and Yahoo prices are revised over time, so a rebuild months later may differ in a few rows; step 4 reports exactly how much.

**Credentials.** You need a Kaggle account and an API token (kaggle.com → Settings → API). Never paste a token into a notebook that you commit or share. Set it as environment variables before starting Jupyter:

```bash
export KAGGLE_USERNAME=your_username        # Windows PowerShell: $env:KAGGLE_USERNAME="your_username"
export KAGGLE_KEY=your_api_key              #                     $env:KAGGLE_KEY="your_api_key"
```

or place the `kaggle.json` file downloaded from Kaggle in `~/.kaggle/` (Windows: `C:\Users\<you>\.kaggle\`).

In [ ]:
# fmflow-setup: runs as-is everywhere. In an installed environment it only locates the repository; on Google Colab,
# Binder or a Jupyter without the package, it downloads the repository once and installs it (about a minute).
import importlib, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get("FMFLOW_REPO_URL", "https://github.com/bahakh19/first-mover-flow")   # set it to use a fork


def _run(cmd: list, what: str, hint: str = "") -> None:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{what} failed:\n{(r.stderr or r.stdout).strip()[-1500:]}\n{hint}")


_here = Path.cwd()
_root = next((p for p in (_here, *_here.parents, _here / "first-mover-flow") if (p / "src" / "fmflow").is_dir()), None)
if _root is None:                                                  # notebooks need the repository's data/ folder
    _root = (_here / "first-mover-flow").resolve()
    print(f"Downloading {REPO_URL} ...")
    if shutil.which("git"):
        _run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(_root)], f"git clone {REPO_URL}",
             "Is the repository public? For a fork or a local copy, set the environment variable FMFLOW_REPO_URL.")
    else:                                                          # no git: the GitHub zip archive of the main branch
        import io, urllib.request, zipfile
        with urllib.request.urlopen(REPO_URL.rstrip("/") + "/archive/refs/heads/main.zip") as _r:
            zipfile.ZipFile(io.BytesIO(_r.read())).extractall(_here)
        (_here / (REPO_URL.rstrip("/").split("/")[-1] + "-main")).rename(_root)
if importlib.util.find_spec("fmflow") is None:
    print("Installing fmflow and its dependencies (about a minute) ...")
    _target = f"{_root}[plot,data]"
    if importlib.util.find_spec("pip"):
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", _target], "pip install")
    elif shutil.which("uv"):                                       # uv-created environments have no pip
        _run(["uv", "pip", "install", "-q", "--python", sys.executable, "-e", _target], "uv pip install")
    else:
        raise RuntimeError("this kernel has neither pip nor uv: install the package from a terminal (INSTALL.md)")
    sys.path.insert(0, str(_root / "src"))
    importlib.invalidate_caches()
os.chdir(_root / "notebooks")
print("fmflow", importlib.import_module("fmflow").__version__, "| repository:", _root)

In [ ]:
# Requirements for this notebook only (not needed by the fmflow package):
# pip install kaggle yfinance pyarrow
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from fmflow.data.normalize import compare_tables, normalize_universe

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"            # git-ignored
REBUILT = ROOT / "data" / "rebuilt"    # git-ignored
RAW.mkdir(parents=True, exist_ok=True)
REBUILT.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "yijian2427/us-market-analyst-insights-with-full-stock-symbols"
PARQUET_FILE = RAW / "analyst_insights.parquet"

## 1. Kaggle credentials and download

The token is read from the environment (or an existing `~/.kaggle/kaggle.json`); nothing is written into this notebook.

In [ ]:
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
user, key = os.environ.get("KAGGLE_USERNAME", "").strip(), os.environ.get("KAGGLE_KEY", "").strip()
if user and key and not kaggle_json.exists():
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_text(json.dumps({"username": user, "key": key}))
    try:
        os.chmod(kaggle_json, 0o600)
    except OSError:
        pass                                  # Windows: permissions are handled by the user profile
if not (kaggle_json.exists() or (user and key) or os.environ.get("KAGGLE_API_TOKEN")):
    raise RuntimeError("No Kaggle credentials: set KAGGLE_USERNAME and KAGGLE_KEY, or put kaggle.json in ~/.kaggle/")

if not PARQUET_FILE.exists():
    print(f"Downloading {DATASET_SLUG} ...")
    cmd = [sys.executable, "-m", "kaggle", "datasets", "download", "-d", DATASET_SLUG, "-p", str(RAW), "--unzip"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-2000:], res.stderr[-2000:])
    res.check_returncode()
    found = sorted(RAW.glob("*.parquet"))
    if found and found[0] != PARQUET_FILE:
        found[0].rename(PARQUET_FILE)
assert PARQUET_FILE.exists(), f"Cannot find {PARQUET_FILE}"

df_raw = pd.read_parquet(PARQUET_FILE)
print(f"Raw parquet shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()

## 2. Target prices for the 30-stock universe

The dataset reports each rating at the **broker** level (`firm`), not the individual analyst. In the open dataset an "analyst" is therefore a brokerage; leaders and herders are brokers.

**How the universe was chosen.** It is the ten development names plus twenty of the most liquid, most covered US large caps in technology, communication services, consumer and financials, chosen by the author. The list is written below, before any return is computed. Two consequences are documented in DATA_SOURCES.md: the out-of-sample period (2024–2026) never overlaps the development period (2016–2021), but **ten of the thirty names are the development names**; and choosing today's large caps is a selection that knows which companies are large today.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower()
col_map = {}
for c in df.columns:
    if c in ("date", "published_date", "publisheddate", "rating_date"):
        col_map[c] = "date"
    elif c in ("symbol", "ticker", "stock"):
        col_map[c] = "stock"
    elif c in ("price_target", "target_price", "target", "estimate_value", "pt"):
        col_map[c] = "estimate_value"
    elif c in ("firm", "broker", "analyst", "analyst_name"):
        col_map[c] = "analyst"
df = df.rename(columns=col_map)
assert {"date", "stock", "estimate_value", "analyst"} <= set(df.columns), df.columns.tolist()

DEVELOPMENT_NAMES = ["AAPL", "BA", "CHGG", "INTC", "JPM", "MSFT", "NKE", "NVDA", "PANW", "QCOM"]
EXPANSION_NAMES = ["GOOGL", "AMZN", "META", "TSLA", "AMD", "AVGO", "CRM", "NFLX", "ORCL", "CSCO",
                   "TXN", "ADBE", "IBM", "GS", "MS", "UNH", "V", "MA", "HD", "COST"]
TARGET_UNIVERSE = DEVELOPMENT_NAMES + EXPANSION_NAMES

r = df.loc[df["stock"].astype(str).str.upper().str.strip().isin(TARGET_UNIVERSE)].copy()
r["stock"] = r["stock"].astype(str).str.upper().str.strip()
r["date"] = pd.to_datetime(r["date"], errors="coerce").dt.strftime("%Y-%m-%d")
r["estimate_value"] = pd.to_numeric(r["estimate_value"], errors="coerce")
r["analyst"] = r["analyst"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

# quality filters (the fmflow cleaning rules R1-R6 run again on ingestion)
r = r.dropna(subset=["date", "estimate_value", "stock", "analyst"])
r = r[r["estimate_value"] > 0]
r = r.drop_duplicates(subset=["date", "stock", "analyst"], keep="last")      # one target per broker, stock and day
r = r.sort_values(["date", "stock", "analyst"]).reset_index(drop=True)
ratings_raw = r[["date", "estimate_value", "stock", "analyst"]]

print(f"target prices: {len(ratings_raw):,} | stocks: {ratings_raw.stock.nunique()} | brokers: {ratings_raw.analyst.nunique()}")
print(f"dates: {ratings_raw.date.min()} to {ratings_raw.date.max()}")
ratings_raw.stock.value_counts().head(10)

## 3. Closing prices

`Close` with `auto_adjust=False` is the closing price **adjusted for splits but not for dividends** in `yfinance`. Kaggle target prices are on the same split-adjusted basis: for the two stocks that split 10-for-1 inside the window (NVDA in June 2024, AVGO in July 2024), the median target / price ratio is between 1.15 and 1.36 both before and after the split (an unadjusted series would jump tenfold). The `end` date of `yf.download` is exclusive, so prices stop one session before the last rating date; ratings published after the last price are dropped by the cleaning rule R6.

In [ ]:
import yfinance as yf

tickers = sorted(ratings_raw["stock"].unique())
px = yf.download(tickers=tickers, start=ratings_raw["date"].min(), end=ratings_raw["date"].max(),
                 auto_adjust=False, progress=False)["Close"]
prices_raw = (px.reset_index().melt(id_vars="Date", var_name="stock", value_name="price")
              .rename(columns={"Date": "date"}))
prices_raw["date"] = pd.to_datetime(prices_raw["date"]).dt.strftime("%Y-%m-%d")
prices_raw["price"] = pd.to_numeric(prices_raw["price"], errors="coerce")
prices_raw = prices_raw.dropna().query("price > 0").sort_values(["date", "stock"]).reset_index(drop=True)

ratings_raw.to_csv(RAW / "Ratings.csv", index=False)
prices_raw.to_csv(RAW / "MarketData.csv", index=False)
print(f"saved {RAW / 'Ratings.csv'} ({len(ratings_raw):,} rows) and {RAW / 'MarketData.csv'} ({len(prices_raw):,} rows)")

## 4. Anonymise, normalise and compare with the shipped files

`normalize_universe` applies three deterministic rules:
* stocks are numbered in alphabetical order of ticker (`AAPL` → `stock_1`, ..., `V` → `stock_30`);
* brokers are numbered in alphabetical order of name (Python string order, upper case before lower case);
* each stock's prices **and** target prices are divided by its first closing price and rounded to 4 decimals.

Every signal uses ratios (returns, target / price, target / previous target), so the normalisation changes no signal. It hides price levels and names. The mapping keys stay in `data/raw/` and are never shipped.

In [ ]:
u = normalize_universe(prices_raw, ratings_raw, decimals=4)
u.prices.to_csv(REBUILT / "MarketData_Normalized.csv", index=False)
u.ratings.to_csv(REBUILT / "Ratings_Normalized.csv", index=False)
u.stock_map.rename_axis("ticker").to_csv(RAW / "stock_map.csv")
u.analyst_map.rename_axis("broker").to_csv(RAW / "analyst_map.csv")

shipped_p = pd.read_csv(ROOT / "data/open/MarketData_Normalized.csv")
shipped_r = pd.read_csv(ROOT / "data/open/Ratings_Normalized.csv")
cmp_p = compare_tables(u.prices, shipped_p, ["date", "stock"], "price")
cmp_r = compare_tables(u.ratings, shipped_r, ["date", "stock", "analyst"], "estimate_value")
pd.DataFrame({"prices": cmp_p, "ratings": cmp_r})

**Reading the comparison.** `only_in_a` / `only_in_b` count rows present in the rebuild but not in the shipped file, or the reverse; `max_abs_diff` is the largest difference in normalised value for matching rows; `share_within_tol` the share of matching rows that agree to rounding precision. A fresh download should agree on essentially every row. Differences point to revisions by the data providers (Kaggle updates, Yahoo corrections) after the frozen files were built, not to a different procedure. The pre-registered test always uses the frozen files in `data/open/`.

To run the strategy on the rebuilt files instead:

```python
from fmflow import StrategyConfig, load_prices, load_ratings, run_strategy
P = load_prices(REBUILT / "MarketData_Normalized.csv")
R = load_ratings(REBUILT / "Ratings_Normalized.csv", P)
out = run_strategy(P, R, StrategyConfig.fmf_n_scalable())
```

**Author's run** (the one that produced the shipped files): the Kaggle file had 53,776 rows and 17 columns; the 30 tickers gave 2,373 target prices from 51 brokers between 2023-01-04 and 2026-08-06, and Yahoo Finance gave 26,970 price rows (899 sessions × 30 stocks) up to 2026-08-05.